전체 흐름\
전체 stock history -> ticker별 과거 시계열 ->
return
momentum
volatility
drawdown
liquidity
size
->
weekly signal date만 추출
->
investable universe 50과 join
->
asset feature dataset

In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\code\portfolio_optimization")

print(PROJECT_ROOT)

C:\code\portfolio_optimization


In [23]:
# 05a4-1. 데이터 경로 설정

KRX_STOCK_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "krx"
    / "stocks"
    / "krx_stock_panel_clean.parquet"
)

INVESTABLE_UNIVERSE_PATH = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "krx"
    / "universe"
    / "krx300_weekly_investable_50.parquet"
)

In [24]:
# 05a4-2. asset feature용 stock 데이터 불러오기

import numpy as np
import pyarrow.parquet as pq

feature_columns = [
    "date",
    "ticker",
    "close",
    "change_pct",
    "volume",
    "trading_value",
    "market_cap",
    "is_no_trade",
    "is_invalid_ohlc"
]

stock_table = pq.read_table(
    KRX_STOCK_PANEL_PATH,
    columns=feature_columns
)

stock_features = (
    stock_table
    .to_pandas()
    .sort_values(
        [
            "ticker",
            "date"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    stock_features.shape
)

(5278578, 9)


In [25]:
# 05a4-3. 종목별 일간 수익률 계산
# krx change_pct 사용

stock_features["return_1d"] = (
    stock_features["change_pct"]
    / 100
)

print(
    stock_features["return_1d"].describe()
)

count    5.278578e+06
mean     3.054765e-04
std      1.354515e-01
min     -9.950000e-01
25%     -1.380000e-02
50%      0.000000e+00
75%      1.080000e-02
max      2.994808e+02
Name: return_1d, dtype: float64


In [26]:
# 05a4-4. 누적 수익률과 momentum 계산

def rolling_compound(series, window):

    return (
        (1 + series)
        .rolling(
            window=window,
            min_periods=window
        )
        .apply(
            np.prod,
            raw=True
        )
        - 1
    )


stock_features["return_5d"] = (
    stock_features
    .groupby(
        "ticker",
        sort=False
    )
    ["return_1d"]
    .transform(
        lambda x: rolling_compound(
            x,
            5
        )
    )
)


stock_features["momentum_20d"] = (
    stock_features
    .groupby(
        "ticker",
        sort=False
    )
    ["return_1d"]
    .transform(
        lambda x: rolling_compound(
            x,
            20
        )
    )
)


stock_features["momentum_60d"] = (
    stock_features
    .groupby(
        "ticker",
        sort=False
    )
    ["return_1d"]
    .transform(
        lambda x: rolling_compound(
            x,
            60
        )
    )
)

In [27]:
# 05a4-5. 20일 volatility 계산

stock_features["volatility_20d"] = (
    stock_features
    .groupby(
        "ticker",
        sort=False
    )
    ["return_1d"]
    .rolling(
        window=20,
        min_periods=20
    )
    .std()
    .reset_index(
        level=0,
        drop=True
    )
    * (252 ** 0.5)
)

일간 volatility × √252= 연율화 volatility

In [28]:
# 05a4-6. return index 계산

stock_features["return_index"] = (
    stock_features
    .groupby(
        "ticker",
        sort=False
    )
    ["return_1d"]
    .transform(
        lambda x: (
            1 + x
        ).cumprod()
    )
)

In [29]:
# 05a4-7. 20일 drawdown 계산

rolling_max_20 = (
    stock_features
    .groupby(
        "ticker",
        sort=False
    )
    ["return_index"]
    .rolling(
        window=20,
        min_periods=20
    )
    .max()
    .reset_index(
        level=0,
        drop=True
    )
)


stock_features["drawdown_20d"] = (
    stock_features["return_index"]
    / rolling_max_20
    - 1
)

In [30]:
# 05a4-8. 20일 평균 거래대금 계산

stock_features["trading_value_ma20"] = (
    stock_features
    .groupby(
        "ticker",
        sort=False
    )
    ["trading_value"]
    .rolling(
        window=20,
        min_periods=20
    )
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
)

In [31]:
# 05a4-9. 거래대금 비율 계산

stock_features["trading_value_ratio_20d"] = (
    stock_features["trading_value"]
    / stock_features["trading_value_ma20"]
)

In [32]:
# 05a4-10. 시가총액 log 변환

stock_features["log_market_cap"] = np.log(
    stock_features["market_cap"]
)

In [33]:
# 05a4-11. asset feature 목록 설정

ASSET_FEATURES = [
    "return_1d",
    "return_5d",
    "momentum_20d",
    "momentum_60d",
    "volatility_20d",
    "drawdown_20d",
    "trading_value_ma20",
    "trading_value_ratio_20d",
    "log_market_cap"
]

print(
    "feature count:",
    len(ASSET_FEATURES)
)

feature count: 9


In [34]:
# 05a4-12. updated investable universe 불러오기

investable_table = pq.read_table(
    INVESTABLE_UNIVERSE_PATH
)

investable_universe = (
    investable_table
    .to_pandas()
)

print(
    "rows:",
    len(investable_universe)
)

print(
    "signals:",
    investable_universe[
        "signal_date"
    ]
    .nunique()
)

print(
    "start:",
    investable_universe[
        "signal_date"
    ].min()
)

print(
    "end:",
    investable_universe[
        "signal_date"
    ].max()
)

rows: 21900
signals: 438
start: 2018-05-04 00:00:00
end: 2026-09-15 00:00:00


In [35]:
# 05a4-13. weekly signal asset feature 연결

signal_asset_features = (
    stock_features[
        [
            "date",
            "ticker"
        ]
        + ASSET_FEATURES
    ]
    .rename(
        columns={
            "date": "signal_date"
        }
    )
)


weekly_asset_features = (
    investable_universe[
        [
            "signal_date",
            "execution_date",
            "ticker",
            "name",
            "market",
            "liquidity_rank"
        ]
    ]
    .merge(
        signal_asset_features,
        on=[
            "signal_date",
            "ticker"
        ],
        how="left"
    )
)


print(
    "shape:",
    weekly_asset_features.shape
)

print(
    "signals:",
    weekly_asset_features[
        "signal_date"
    ].nunique()
)

shape: (21900, 15)
signals: 438


In [36]:
# 05a4-14. feature nan 확인

print(
    weekly_asset_features[
        ASSET_FEATURES
    ]
    .isna()
    .sum()
)

return_1d                  0
return_5d                  0
momentum_20d               0
momentum_60d               0
volatility_20d             0
drawdown_20d               0
trading_value_ma20         0
trading_value_ratio_20d    0
log_market_cap             0
dtype: int64


In [37]:
# 05a4-15. feature inf 확인

inf_summary = pd.Series({
    col: np.isinf(
        weekly_asset_features[col]
    ).sum()
    for col in ASSET_FEATURES
})

print(
    inf_summary
)

return_1d                  0
return_5d                  0
momentum_20d               0
momentum_60d               0
volatility_20d             0
drawdown_20d               0
trading_value_ma20         0
trading_value_ratio_20d    0
log_market_cap             0
dtype: int64


KRX 등락률을 쓰면 액면분할이나 증자 때문에 주가 숫자가 기계적으로 바뀌는 문제는 어느 정도 처리할 수 있다. 하지만 배당금까지 포함한 실제 투자자의 전체 수익률을 의미하는 것은 아니다. 우리는 주로 주가 변화에 따른 수익률을 사용한다.

그냥 종가로 계산
오늘 종가 / 어제 종가 - 1
        ↓
액면분할 같은 날
가짜 -90% 수익률이 생길 수 있음

KRX 등락률 사용
        ↓
Corporate Action에 따른 기준가격 조정 반영
        ↓
주가 움직임을 더 자연스럽게 측정

하지만
        ↓
배당금은 포함하지 않음
        ↓
Price Return 계열
≠ Total Return

모델 feature에 배당을 넣으려면 “그 배당금 정보가 당시 실제로 알려져 있었는가?”까지 point-in-time으로 관리해야 해서 훨씬 복잡. 반면 실제로 투자했을 때 발생한 수익을 평가할 때 배당을 포함하는 건 의미가 있을듯.

현재 KRX Open API의 일별 주식 데이터에는 배당금 필드가 포함돼 있지 x
공식 서비스는 일별 주식 매매정보를 제공, 현재 받은 OHLC·거래량·거래대금·시총 같은 데이터 중심임

open dart -> 배당에 관한 사항 api있음 -> 이걸로 추후에 데이터 구조를 ticker, ex_date, divided_per_share, payment_date 처럼 만들어서 stock panel과 연결할 예정

실제 execution 방식
배당락일 -> 받을 권리만 발생, 지급일 -> cash 유입, 다음 rebalance -> cash까지 포함해서 투자

In [38]:
# 05a4-16. signal별 종목 수 확인

weekly_count = (
    weekly_asset_features
    .groupby(
        "signal_date"
    )
    ["ticker"]
    .nunique()
)


print(
    weekly_count.describe()
)

count    438.0
mean      50.0
std        0.0
min       50.0
25%       50.0
50%       50.0
75%       50.0
max       50.0
Name: ticker, dtype: float64


In [39]:
# 05a4-17. weekly asset feature 저장

import pyarrow as pa
import pyarrow.parquet as pq


ASSET_FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "assets"
)

ASSET_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ASSET_FEATURE_PATH = (
    ASSET_FEATURE_DIR
    / "weekly_asset_features.parquet"
)


asset_table = pa.Table.from_pandas(
    weekly_asset_features,
    preserve_index=False
)


pq.write_table(
    asset_table,
    ASSET_FEATURE_PATH,
    compression="snappy"
)


print(
    "saved:",
    ASSET_FEATURE_PATH
)

saved: C:\code\portfolio_optimization\data\features\assets\weekly_asset_features.parquet


In [40]:
# 05a4-18. 저장 결과 확인

asset_check = (
    pq.read_table(
        ASSET_FEATURE_PATH
    )
    .to_pandas()
)


print(
    "shape:",
    asset_check.shape
)

print(
    "signals:",
    asset_check[
        "signal_date"
    ].nunique()
)

print(
    "tickers:",
    asset_check[
        "ticker"
    ].nunique()
)

print(
    "duplicates:",
    asset_check[
        [
            "signal_date",
            "ticker"
        ]
    ]
    .duplicated()
    .sum()
)

shape: (21900, 15)
signals: 438
tickers: 309
duplicates: 0
